# LabelEngine Validation & Experimentation Workbook

This notebook validates the newly implemented `LabelEngine` in the Forex_DNN trading framework.
The `LabelEngine` is responsible for generating deterministic, rule-based labels for machine learning datasets based on predefined strategic logic from `MarketStructureEngine` and `SupplyDemandEngine`.

## Core Objectives
1. Demonstrate that the **window size is configurable** (testing 20, 25, 35, 50, and 75).
2. Generate labeled datasets and verify that **unlabeled or ambiguous samples are removed**.
3. Run the `DatasetValidator` on each generated dataset to verify consistency, duplicates, and missing values.
4. Verify the **reproducibility manifest** (`.manifest.json`) is saved alongside each dataset.
5. Plot and compare the class distributions across different window sizes using **Matplotlib** exclusively.

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from ML.label_engine import LabelEngine
from ML.market_state_labeler import MarketStateLabeler
from ML.dataset_validator import DatasetValidator
from scripts.generate_dataset import generate_synthetic_ohlcv

print("Libraries imported successfully.")

### Step 1: Generate Synthetic Multi-Regime Data
We generate 1,200 bars of synthetic OHLCV data representing various trends, sideways ranges, and volatile states.

In [ ]:
df_raw = generate_synthetic_ohlcv(n_bars=1200)
df_raw.head()

### Step 2: Compare Different Window Sizes
We will run the `LabelEngine` across different window sizes (`20, 25, 35, 50, 75`) with `window_stride = 1` and examine the resulting datasets, manifests, and class distributions.

In [ ]:
window_sizes = [20, 25, 35, 50, 75]
results = {}

os.makedirs("output/experiments", exist_ok=True)

for w_size in window_sizes:
    print(f"\nProcessing window_size = {w_size}...")
    
    # Initialize labeler and engine
    labeler = MarketStateLabeler(min_confidence=0.4)
    engine = LabelEngine(window_size=w_size, window_stride=1)
    
    output_path = f"output/experiments/dataset_w{w_size}.csv"
    
    # Generate
    dataset_df = engine.generate(
        df=df_raw,
        symbol="EURUSD",
        timeframe="M5",
        labeler=labeler,
        output_csv_path=output_path
    )
    
    # Validate
    validator = DatasetValidator(expected_window_size=w_size)
    val_report = validator.validate(dataset_df)
    
    # Read the manifest to verify saving
    manifest_path = output_path + ".manifest.json"
    with open(manifest_path, "r") as f:
        manifest = json.load(f)
        
    results[w_size] = {
        "retained_samples": len(dataset_df),
        "removed_samples": manifest["samples_removed"]["total"],
        "is_valid": val_report["is_valid"],
        "class_distribution": manifest["final_class_distribution"],
        "manifest_reproducible": "feature_registry_version" in manifest
    }
    
print("\nAll experiments completed successfully.")

### Step 3: Present Experimentation Results Summary
Let's print out a structured table detailing retained samples and validity of every experiment.

In [ ]:
summary_data = []
for w_size, data in results.items():
    dist = data["class_distribution"]
    trend_cnt = dist.get("TREND", {}).get("count", 0)
    range_cnt = dist.get("RANGE", {}).get("count", 0)
    trans_cnt = dist.get("TRANSITION", {}).get("count", 0)
    
    summary_data.append({
        "Window Size": w_size,
        "Retained Samples": data["retained_samples"],
        "Removed Samples": data["removed_samples"],
        "TREND Count": trend_cnt,
        "RANGE Count": range_cnt,
        "TRANSITION Count": trans_cnt,
        "Valid": data["is_valid"],
        "Manifest Saved": data["manifest_reproducible"]
    })

summary_df = pd.DataFrame(summary_data)
summary_df

### Step 4: Visualize Class Distributions via Matplotlib
We plot side-by-side comparative bar charts showing label counts across different window sizes.

In [ ]:
window_labels = [f"W-{w}" for w in window_sizes]
trend_counts = [results[w]["class_distribution"].get("TREND", {}).get("count", 0) for w in window_sizes]
range_counts = [results[w]["class_distribution"].get("RANGE", {}).get("count", 0) for w in window_sizes]
trans_counts = [results[w]["class_distribution"].get("TRANSITION", {}).get("count", 0) for w in window_sizes]

x = np.arange(len(window_labels))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))
rects1 = ax.bar(x - width, trend_counts, width, label='TREND', color='#4F46E5')
rects2 = ax.bar(x, range_counts, width, label='RANGE', color='#10B981')
rects3 = ax.bar(x + width, trans_counts, width, label='TRANSITION', color='#F59E0B')

ax.set_ylabel('Sample Counts')
ax.set_title('Class Distribution comparison Across Window Sizes')
ax.set_xticks(x)
ax.set_xticklabels(window_labels)
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.7)

fig.tight_layout()
plt.savefig("output/experiments/class_distribution_comparison.png", dpi=150)
plt.show()

### Step 5: Verify the Dataset Manifest Format
Let's load the generated manifest of window size `35` and verify its reproducibility fields.

In [ ]:
with open("output/experiments/dataset_w35.csv.manifest.json", "r") as f:
    manifest_35 = json.load(f)

print("--- Manifest Verification Sample (window_size=35) ---")
print(json.dumps(manifest_35, indent=2))